<a href="https://colab.research.google.com/github/Leo278V/Final-Assignment-PDS/blob/Final-Assignment-V2/NLP_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [160]:
import pandas as pd

In [161]:
data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/linkedin_experience_annotated.csv"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [162]:
#Fill active Jobs with current date
from datetime import date


df_profiles["endDate"] = df_profiles["endDate"].astype(str)

df_profiles.loc[
    ((df_profiles["status"] == "ACTIVE") | (df_profiles["status"] == "UNKNOWN")) & (df_profiles["endDate"].isin(["nan", "NaT"])),
    "endDate"
] = date.today().strftime("%Y-%m")

In [163]:
#Remove linkedIN  (URL) column

df_profiles.drop(columns=['linkedin'], inplace=True, errors='ignore')
df_profiles.head()

,organization,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0


In [164]:
# Imputing strategy for Unknown and missing startDate where there is only one (current job)

df_profiles["startDate"] = df_profiles["startDate"].replace(
    ["", "unknown", "novalue", None],
    pd.NA
)

job_counts = df_profiles.groupby("person_id").size()
df_profiles["job_count"] = df_profiles["person_id"].map(job_counts)

# Calculate job_duration_years here before using it
# Use errors='coerce' to handle non-conforming date strings gracefully
start_temp = pd.to_datetime(df_profiles["startDate"], format="%Y-%m", errors='coerce')
end_temp   = pd.to_datetime(df_profiles["endDate"],   format="%Y-%m", errors='coerce')
df_profiles["job_duration_years"] = (end_temp - start_temp).dt.days / 365

#Median Job Duration for imputing
median_duration_years = df_profiles["job_duration_years"].median()
median_duration_months = int(round(median_duration_years * 12))

mask = (
    df_profiles["startDate"].isna() & # Corrected 'df' to 'df_profiles'
    (df_profiles["job_count"] == 1)
)

# endDate als Referenz, sonst heutiges Datum
reference_date = pd.to_datetime(
    df_profiles.loc[mask, "endDate"], # Corrected 'df' to 'df_profiles'
    format="%Y-%m",
    errors="coerce"
).fillna(pd.Timestamp.today())

imputed_start = reference_date - pd.DateOffset(months=median_duration_months)

df_profiles.loc[mask, "startDate"] = imputed_start.dt.strftime("%Y-%m")

In [165]:
#Drop Rest where theres no start Date and more than one job count (only 23)

df_profiles = df_profiles.drop(
    df_profiles[
        df_profiles["startDate"].isna() &
        (df_profiles["job_count"] > 1)
    ].index
)
#and change status from unknown for imputed to active
df_profiles.loc[
    (df_profiles["status"].str.lower() == "unknown"),
    "status"
] = "ACTIVE"

In [166]:
# Ready to use dataset for Encoding

df_profiles.head()

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658


In [167]:
#Regularize startDate for job_duration_years
import pandas as pd
import re

def normalize_startdate(val):
    if pd.isna(val):
        return pd.NA

    val = str(val).strip()

    # Case 1: YYYY-MM (bereits korrekt)
    if re.fullmatch(r"\d{4}-\d{2}", val):
        return val

    # Case 2: YYYY \u2192 erg\u00e4nze Januar
    if re.fullmatch(r"\d{4}", val):
        return f"{val}-01"

    # alles andere \u2192 missing
    return pd.NA


df_profiles["startDate"] = df_profiles["startDate"].apply(normalize_startdate)

import pandas as pd
import re

def normalize_enddate(val):
    if pd.isna(val):
        return pd.NA

    val = str(val).strip()

    # Case 1: YYYY-MM (bereits korrekt)
    if re.fullmatch(r"\d{4}-\d{2}", val):
        return val

    # Case 2: YYYY → ergänze Januar
    if re.fullmatch(r"\d{4}", val):
        return f"{val}-01"

    # alles andere → missing
    return pd.NA


df_profiles["endDate"] = df_profiles["endDate"].apply(normalize_enddate)


In [168]:
#Check for missing values
df_profiles.isna().sum()

,0
organization,0
position,0
startDate,0
endDate,0
status,0
department,0
seniority,0
person_id,0
job_count,0
job_duration_years,325


In [169]:
#Did we successfully change all Unknown to Active if theres only one job?
df_profiles["status"].value_counts(dropna=False)

,count
status,
INACTIVE,1897
ACTIVE,718


In [170]:
#Ready to encode dataframe:

df_profiles.head(75)

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
...,...,...,...,...,...,...,...,...,...,...
70,Lichtenberg School,"Teacher: History, Ethics, French",2008-08,2017-07,INACTIVE,Other,Professional,20,6,8.920548
71,German School Beijing (China),"Teacher: History, Ethics, French",2002-08,2008-07,INACTIVE,Other,Professional,20,6,5.920548
72,"Universities: Chuncheon, Hannam, Hongik (South...",Dr. phil. - German Studies,1990-02,1999-01,INACTIVE,Other,Professional,20,6,8.920548
73,Thurm GmbH,"Geschäftsführer, CMO",2014-07,2025-12,ACTIVE,Marketing,Management,21,3,11.427397


In [179]:
#Download Code for Team Members
#df_profiles.to_csv("df_seniority_cleansed.csv", index=False)

#from google.colab import files
#files.download("df_seniority_cleansed.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Feature Encoding

In [171]:
df_encoded = df_profiles.copy()

In [172]:
#Organization encoding - Frequency Encoding

org_freq = df_encoded["organization"].value_counts(normalize=True)
df_encoded["organization_freq"] = df_encoded["organization"].map(org_freq)

df_encoded = df_encoded.drop(columns=["organization"])

In [173]:
#Position Encoding - Sentence Embeddings

from sentence_transformers import SentenceTransformer
import pandas as pd # Ensure pandas is imported

model = SentenceTransformer("all-MiniLM-L6-v2")
position_embeddings = model.encode(df_encoded["position"].fillna("").tolist())

# Create new column names for the embeddings
embedding_column_names = [f"position_embed_{i}" for i in range(position_embeddings.shape[1])]

# Convert embeddings to a DataFrame and align with df_encoded's index
position_embeddings_df = pd.DataFrame(position_embeddings, index=df_encoded.index, columns=embedding_column_names)

# Drop the original 'position' column and concatenate the new embedding columns
df_encoded = df_encoded.drop(columns=["position"])
df_encoded = pd.concat([df_encoded, position_embeddings_df], axis=1)

In [174]:
#startDate & endDate -> Feature Engineering → job_duration_years
start = pd.to_datetime(df_encoded["startDate"], format="%Y-%m", errors='coerce')
end   = pd.to_datetime(df_encoded["endDate"],   format="%Y-%m", errors='coerce')

df_encoded["job_duration_years"] = (end - start).dt.days / 365

df_encoded = df_encoded.drop(columns=["startDate", "endDate"])

In [175]:
#status → Binary Encoding
df_encoded["status_bin"] = df_encoded["status"].map({"ACTIVE": 1, "INACTIVE": 0})
df_encoded = df_encoded.drop(columns=["status"])


In [176]:
df_encoded.head()

,department,seniority,person_id,job_count,job_duration_years,organization_freq,position_embed_0,position_embed_1,position_embed_2,position_embed_3,...,position_embed_375,position_embed_376,position_embed_377,position_embed_378,position_embed_379,position_embed_380,position_embed_381,position_embed_382,position_embed_383,status_bin
0,Other,Management,0,6,6.339726,0.001912,-0.038326,0.013078,-0.139571,-0.024655,...,-0.009933,0.025318,0.005560,-0.074910,0.005907,0.137119,-0.000993,0.093262,0.003507,1
1,Other,Management,0,6,6.424658,0.001912,-0.059618,-0.047214,-0.039971,0.075319,...,-0.063512,-0.044259,0.009971,-0.051646,-0.010528,-0.038092,-0.017355,0.083279,0.003473,1
2,Other,Professional,0,6,6.424658,0.001912,-0.048715,0.004592,-0.039356,-0.007562,...,0.019506,0.009547,0.036486,-0.109967,0.016005,-0.005102,-0.030709,0.058480,-0.000570,1
3,Other,Management,0,6,6.424658,0.001912,-0.052659,0.031287,-0.152101,-0.028479,...,0.010195,0.039245,0.024023,-0.046679,0.002175,0.115020,-0.004019,0.089740,0.000141,1
4,Other,Management,0,6,6.424658,0.001912,-0.059618,-0.047214,-0.039971,0.075319,...,-0.063512,-0.044259,0.009971,-0.051646,-0.010528,-0.038092,-0.017355,0.083279,0.003473,1


In [177]:
print(train_X.info())
display(train_X.head())

<class 'pandas.core.frame.DataFrame'>
Index: 1961 entries, 1774 to 2630
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   position            1961 non-null   object 
 1   person_id           1961 non-null   int64  
 2   job_count           1961 non-null   int64  
 3   job_duration_years  1961 non-null   float64
 4   organization_freq   1961 non-null   float64
 5   status_bin          1961 non-null   int64  
 6   department_enc      1961 non-null   int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 122.6+ KB
None


,position,person_id,job_count,job_duration_years,organization_freq,status_bin,department_enc
1774,AUTOMATICIEN ELECTRICIEN,397,2,17.928767,0.000382,1,7
230,Managing Partner,59,16,6.583562,0.000382,0,1
508,Director of Sales,122,10,2.331507,0.000382,0,10
2394,Director Projects,556,7,6.758904,0.000765,1,8
860,"COO, Supply Chain Director, Co-owner",200,16,1.832877,0.001530,0,7


In [195]:
df_seniority_raw = df_encoded[embedding_column_names + ["seniority", "department"]
].copy()



**Datframes for Seniority and Department**

In [197]:
from sklearn.preprocessing import LabelEncoder

seniority_map = {
    "Junior": 0,
    "Professional": 1,
    "Senior": 2,
    "Lead": 3,
    "Management": 4,
    "Director": 5
}

df_seniority_encoded = df_seniority_raw[
  embedding_column_names + ["seniority", "department"]
].copy()

# Target (Ordinal)
df_seniority_encoded["seniority"] = df_seniority_encoded["seniority"].map(seniority_map)


# Department als Feature (Label Encoding)
le_dept = LabelEncoder()
df_seniority_encoded["department_enc"] = le_dept.fit_transform(
    df_seniority_encoded["department"]
)

df_seniority_encoded = df_seniority_encoded.drop(columns=["department"])

In [201]:
#Department Dataframe for Predicting Department Model
df_department_raw = df_encoded[embedding_column_names +[
    "department",
    "seniority",

]].copy()

df_department_encoded = df_encoded[embedding_column_names +[
    "department",
    "seniority",
]].copy()

# Target (Label Encoding)
le_dep = LabelEncoder()
df_department_encoded["department"] = le_dep.fit_transform(
    df_department_encoded["department"]
)

# Seniority als Feature (Ordinal)
df_department_encoded["seniority_ord"] = df_department_encoded["seniority"].map(seniority_map)
df_department_encoded = df_department_encoded.drop(columns=["seniority"])


Model Predicting Seniority

In [202]:
y = df_seniority_encoded['seniority']

In [203]:
X = df_seniority_encoded.drop('seniority', axis=1)

In [204]:
from sklearn.model_selection import train_test_split


In [205]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)


In [206]:
# Random Forest Model

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(train_X, train_y)


RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=10, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [208]:
# Evaluating Model

from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf_model.predict(val_X)

print(classification_report(val_y, y_pred))


              precision    recall  f1-score   support

           0       0.89      0.54      0.67        61
           1       0.80      0.90      0.85       312
           2       0.94      0.82      0.88        39
           3       0.58      0.67      0.62       107
           4       0.92      0.78      0.85        93
           5       1.00      0.74      0.85        42

    accuracy                           0.80       654
   macro avg       0.86      0.74      0.79       654
weighted avg       0.81      0.80      0.80       654

